# Session 24: The parallelism inside the tools

Last time our own pool of processes kept many cores busy. But NumPy and scikit-learn already start threads of their
own, without asking. Today we find out which calls do it, how to see and set the number of threads, and what happens
when a pool of processes meets a library that starts threads in every worker.

- **Part 1**: which NumPy calls use threads?
- **Part 2**: seeing and setting the threads.
- **Part 3**: the two knobs of scikit-learn.
- **Part 4**: too many threads.
- **Part 5**: homework.

Run the cells in order, from top to bottom: click a cell and press Shift+Enter, on Mac and on Windows. When the text
says **Predict**, decide on your answer before you run the next cell.

## Before you start

Look at the top right corner of this notebook. It must say **OSPC (Python 3.12)**. If it says something else, click
on that name and choose OSPC (Python 3.12) from the list.

The measurements run as batch jobs with 8 CPUs (4 cores) or 4 CPUs (2 cores). Your JupyterLab session has 2 CPUs, one
core, which is too few to see threads at work. The next cell puts the course Python in the `PATH` for the jobs.

In [ ]:
import os
import subprocess
import sys

os.environ["PATH"] = "/data/software/envs/ospc/bin:" + os.environ["PATH"]
print("CPUs of your session:", len(os.sched_getaffinity(0)))

## Part 1. Which NumPy calls use threads?

NumPy hands linear algebra to a library called **BLAS** (here OpenBLAS), which splits a matrix product into blocks
and gives each block to a thread. Other operations, such as adding two arrays or sorting, run in one thread.

The next cell writes `calls.py`. It times five calls, first with one thread and then with all the CPUs of the job.
`threadpool_limits(t)` sets the number of threads of the libraries for the lines inside the `with` block.

**Predict**: which of the five calls will be faster with 8 threads?

In [ ]:
%%writefile calls.py
import os
import time

import numpy as np
from threadpoolctl import threadpool_limits

cpus = len(os.sched_getaffinity(0))
rng = np.random.default_rng(1)
a = rng.random((2000, 2000))
b = rng.random(2000)
x = rng.random(20_000_000)
y = rng.random(20_000_000)

calls = {
    "a @ a": lambda: a @ a,
    "np.linalg.solve(a, b)": lambda: np.linalg.solve(a, b),
    "x + y": lambda: x + y,
    "x.sum()": lambda: x.sum(),
    "np.sort(x)": lambda: np.sort(x),
}


def best_of_3(fn):
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)


print(f"node {os.uname().nodename}, {cpus} CPUs")
print(f"{'call':24s} {'1 thread':>10s} {str(cpus) + ' threads':>11s}")
for name, fn in calls.items():
    with threadpool_limits(1):
        one = best_of_3(fn)
    with threadpool_limits(cpus):
        many = best_of_3(fn)
    print(f"{name:24s} {one:9.3f}s {many:10.3f}s")

In [ ]:
!sbatch -p cpu -c 8 --mem=4G -t 5 -o calls.out --wait --wrap "python calls.py"
!cat calls.out

The matrix product and `solve` use the threads: on 8 CPUs, which are 4 cores, they run about 3 to 4 times faster.
Adding, summing and sorting take the same time with 1 thread and with 8: those calls never start threads.

## Part 2. Seeing and setting the threads

`threadpool_info()` lists the thread pools that the libraries loaded in this Python process, and how many threads each
one will use. Here, in your 2-CPU session:

In [ ]:
import numpy
from threadpoolctl import threadpool_info

for pool in threadpool_info():
    print(pool["user_api"], pool["internal_api"], pool["num_threads"], "threads")

There are two ways to set the number of threads:

- **Environment variables**, read when the library starts: `OMP_NUM_THREADS` (OpenMP, used by scikit-learn),
  `OPENBLAS_NUM_THREADS` (OpenBLAS, used by NumPy). In a job script: `export OMP_NUM_THREADS=1`.
- **In the code**: `with threadpool_limits(1):`, as in Part 1.

**Predict**: the cell below starts three new Python processes. The first does not set the variable. The second sets
`OPENBLAS_NUM_THREADS=1` before `import numpy`, the third after. How many threads will NumPy use in each?

In [ ]:
child = '''
import os, sys
if sys.argv[1] == "before":
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
import numpy
from threadpoolctl import threadpool_info
if sys.argv[1] == "after":
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
print(f"{sys.argv[1]:7s}", [p["num_threads"] for p in threadpool_info() if p["user_api"] == "blas"], "BLAS threads")
'''
for when in ["never", "before", "after"]:
    subprocess.run([sys.executable, "-c", child, when])

Setting the variable after `import numpy` changes nothing: OpenBLAS read it when it started. Set the variables in the
job script, before Python starts, or use `threadpool_limits` in the code.

## Part 3. The two knobs of scikit-learn

scikit-learn has two ways to use more CPUs:

- `n_jobs`: how many fits, trees or folds run at the same time, in a pool of workers (joblib).
- **Threads inside one fit**, from OpenMP or BLAS, set with `OMP_NUM_THREADS` or `threadpool_limits`. `n_jobs` does
  not change them.

The next cell writes `knobs.py`: a random forest and a cross-validation with `n_jobs=1` and `n_jobs=8`, and a
HistGradientBoosting model with 1 and 8 OpenMP threads.

**Predict**: which of the three gains most from 8 CPUs?

In [ ]:
%%writefile knobs.py
import os
import time

from sklearn.datasets import make_classification
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score
from threadpoolctl import threadpool_limits

cpus = len(os.sched_getaffinity(0))
X, y = make_classification(n_samples=20_000, n_features=30, n_informative=12, random_state=0)


def seconds(fn):
    t0 = time.perf_counter()
    fn()
    return time.perf_counter() - t0


print(f"node {os.uname().nodename}, {cpus} CPUs")
for n in [1, cpus]:
    t = seconds(lambda: RandomForestClassifier(100, max_depth=12, n_jobs=n, random_state=0).fit(X, y))
    print(f"RandomForest, 100 trees, n_jobs={n}: {t:.2f} s")
for n in [1, cpus]:
    t = seconds(lambda: cross_val_score(RandomForestClassifier(25, max_depth=12, random_state=0), X, y, cv=8, n_jobs=n))
    print(f"cross_val_score, 8 folds, n_jobs={n}: {t:.2f} s")
for threads in [1, cpus]:
    with threadpool_limits(threads):
        t = seconds(lambda: HistGradientBoostingClassifier(max_iter=100, random_state=0).fit(X, y))
    print(f"HistGradientBoosting, {threads} OpenMP threads: {t:.2f} s")

In [ ]:
!sbatch -p cpu -c 8 --mem=8G -t 5 -o knobs.out --wait --wrap "python knobs.py"
!cat knobs.out

The forest and the cross-validation are many independent fits: `n_jobs` runs them at the same time. HistGradientBoosting
builds its trees one after the other, and each tree uses OpenMP threads inside: there, `n_jobs` would do nothing, and the
threads are the knob.

## Part 4. Too many threads

Now we combine a pool of 4 processes with NumPy's threads, on a job with 4 CPUs. Each process runs matrix products,
and by default OpenBLAS gives each process 4 threads: 4 × 4 = 16 threads on 4 CPUs. This is called
**oversubscription**.

The next cell writes `nested.py`. It runs 16 tasks, each a series of 500 × 500 matrix products, three ways: 4 processes
with 1 thread each, 4 processes with 4 threads each, and joblib with `n_jobs=4`.

**Predict**: which is fastest, and how much slower is the slowest?

In [ ]:
%%writefile nested.py
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

N = int(sys.argv[1]) if len(sys.argv) > 1 else 500
CPUS = len(os.sched_getaffinity(0))
TASKS = 16
PRODUCTS = max(1, 16 * 1000 ** 3 // (2 * N ** 3))    # the same work at every N: 16 billion operations per task


def task(n):
    a = np.ones((n, n))
    for _ in range(PRODUCTS):
        a @ a


def limit(threads):
    threadpool_limits(threads)


def pool(threads):
    with ProcessPoolExecutor(CPUS, initializer=limit, initargs=(threads,)) as ex:
        list(ex.map(task, [N] * TASKS))


def with_joblib():
    Parallel(n_jobs=CPUS)(delayed(task)(N) for _ in range(TASKS))


print(f"node {os.uname().nodename}, {CPUS} CPUs, {TASKS} tasks of {N} x {N} products")
for name, fn in [(f"{CPUS} processes x 1 thread", lambda: pool(1)),
                 (f"{CPUS} processes x {CPUS} threads", lambda: pool(CPUS)),
                 (f"joblib, n_jobs={CPUS}", with_joblib)]:
    t0 = time.perf_counter()
    fn()
    print(f"{name:28s} {time.perf_counter() - t0:6.2f} s")

In [ ]:
!sbatch -p cpu -c 4 --mem=4G -t 5 -o nested.out --wait --wrap "python nested.py 500"
!cat nested.out

With one thread per process, the 4 processes keep the 4 CPUs busy. With 4 threads per process, 16 threads take turns
on 4 CPUs, and every product must wait for the last of its 4 threads: the same work takes several times longer. joblib
avoids the problem on its own: it gives each worker CPUs ÷ `n_jobs` threads, here 1.

The rule for your project: **workers × threads per worker ≤ the CPUs of your job**. Parallelize one level and set the
other to 1.

## Part 5. Homework

1. Run Part 4 with smaller products: change `500` to `200` in the `sbatch` line. **Predict** first: will the
   oversubscribed version suffer more or less than with 500? The job takes about 90 seconds.
2. Write the two job scripts of the slide "Two Job Scripts" for your project, and check with `threadpool_info()`,
   inside the job, that the libraries use the number of threads you set.

In [ ]:
!sbatch -p cpu -c 4 --mem=4G -t 5 -o nested-200.out --wait --wrap "python nested.py 200"
!cat nested-200.out